이 자료는 위키독스 딥 러닝을 이용한 자연어 처리 입문의 코사인 유사도 튜토리얼 자료입니다.  

링크 : https://wikidocs.net/24603

# 1. 코사인 유사도(Cosine Similarity)
- 두 벡터가 이루는 각 $\theta$에 대한 $\cos(\theta)$을 말하며, 두 벡터의 유사도를 의미함
    - 두 벡터의 방향이 완전히 동일한 경우에는 1
    - 90°의 각을 이루면 0
    - 180°로 반대의 방향을 가지면 -1

$$
\def\va{\overrightarrow{A}}
\def\vb{\overrightarrow{B}}
\def\norm#1{\| {#1} \|}

\begin{align*}
\va \cdot \vb &= \norm{\va} \norm{\vb} \cos(\theta)\\
\cos(\theta) &= \frac{\va \cdot \vb}{\norm{\va} \norm{\vb}} = \frac{\sum_{i=1}^n A_i \times B_i}{\sqrt{\sum_{i=1}^n (A_i)^2} \times \sqrt{\sum_{i=1}^n (B_i)^2}}
\end{align*}
$$

- 코사인 유사도는 -1 이상 1 이하의 값을 가지며 값이 1에 가까울수록 유사도가 높음
- 코사인은 각도의 함수이므로 벡터의 크기와 상관없음에 유의!

In [1]:
from numpy import dot
from numpy.linalg import norm
import numpy as np

In [2]:
def cos_sim(A, B):
  return np.round(dot(A, B)/(norm(A)*norm(B)), 4)

In [3]:
doc1 = np.array([0,1,1,1])
doc2 = np.array([1,0,1,1])
doc3 = np.array([2,0,2,2])

In [4]:
print('문서 1과 문서2의 유사도 :',cos_sim(doc1, doc2))
print('문서 1과 문서3의 유사도 :',cos_sim(doc1, doc3))
print('문서 2와 문서3의 유사도 :',cos_sim(doc2, doc3))

문서 1과 문서2의 유사도 : 0.6667
문서 1과 문서3의 유사도 : 0.6667
문서 2와 문서3의 유사도 : 1.0


# 2. 유사도를 이용한 추천 시스템 구현하기
- TF-IDF와 코사인 유사도만으로 영화의 줄거리(`overview`)에 기반해서 영화를 추천하는 추천 시스템 만들기
- [데이터 출처](https://www.kaggle.com/rounakbanik/the-movies-dataset)

## 2.1 데이터 다운로드 및 정제

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
data = pd.read_csv('dataset/movies_metadata.csv', low_memory=False)
print(data.shape)
data.head(10)

(45466, 24)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0
5,False,NaN,60000000,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",NaN,949,tt0113277,en,Heat,"Obsessive master thief, Neil McCauley leads a ...",...,1995-12-15,187436818.0,170.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,A Los Angeles Crime Saga,Heat,False,7.7,1886.0
6,False,NaN,58000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...",NaN,11860,tt0114319,en,Sabrina,An ugly duckling having undergone a remarkable...,...,1995-12-15,0.0,127.0,"[{'iso_639_1': 'fr', 'name': 'Français'}, {'is...",Released,You are cordially invited to the most surprisi...,Sabrina,False,6.2,141.0
7,False,NaN,0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",NaN,45325,tt0112302,en,Tom and Huck,"A mischievous young boy, Tom Sawyer, witnesses...",...,1995-12-22,0.0,97.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,The Original Bad Boys.,Tom and Huck,False,5.4,45.0
8,False,NaN,35000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",NaN,9091,tt0114576,en,Sudden Death,International action superstar Jean Claude Van...,...,1995-12-22,64350171.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Terror goes into overtime.,Sudden Death,False,5.5,174.0
9,False,"{'id': 645, 'name': 'James Bond Collection', '...",58000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.mgm.com/view/movie/757/Goldeneye/,710,tt0113189,en,GoldenEye,James Bond must unmask the mysterious head of ...,...,1995-11-16,352194034.0,130.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,No limits. No fears. No substitutes.,GoldenEye,False,6.6,1194.0


In [7]:
data = data.head(20000)

In [8]:
print('overview 열의 결측값의 수:',data['overview'].isnull().sum())

overview 열의 결측값의 수: 135


In [9]:
data = data[data['overview'].notnull()].reset_index(drop=True)
print(data.shape)
print('overview 열의 결측값의 수:', data['overview'].isnull().sum())

(19865, 24)
overview 열의 결측값의 수: 0


## 2.2 TF-IDF와 코사인 유사도 계산

In [10]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['overview'])
print('TF-IDF 행렬의 크기(shape) :',tfidf_matrix.shape)

TF-IDF 행렬의 크기(shape) : (19865, 47487)


In [11]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [12]:
print('코사인 유사도 연산 결과 :',cosine_sim.shape)

코사인 유사도 연산 결과 : (19865, 19865)


In [13]:
title_to_index = dict(zip(data['title'], data.index))

In [14]:
idx = title_to_index['Father of the Bride Part II']
print(idx)

4


In [15]:
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = title_to_index[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:11]

    movie_indices = [idx[0] for idx in sim_scores]

    return data[['title', 'overview']].iloc[movie_indices]

In [22]:
get_recommendations('The Dark Knight Rises')

,title,overview
12447,The Dark Knight,Batman raises the stakes in his war on crime. ...
149,Batman Forever,The Dark Knight of Gotham City confronts a das...
1314,Batman Returns,"Having defeated the Joker, Batman now faces th..."
15444,Batman: Under the Red Hood,Batman faces his ultimate challenge as the mys...
583,Batman,The Dark Knight of Gotham City begins his war ...
9203,Batman Beyond: Return of the Joker,"The Joker is back with a vengeance, and Gotham..."
17930,Batman: Year One,Two men come to Gotham City: Bruce Wayne after...
19663,"Batman: The Dark Knight Returns, Part 1",Batman has not been seen for ten years. A new ...
3077,Batman: Mask of the Phantasm,An old flame of Bruce Wayne's strolls into tow...
10092,Batman Begins,"Driven by tragedy, billionaire Bruce Wayne ded..."
